# Personal Fit AI — Colab vLLM 서버

Google Colab에서 vLLM + Qwen2.5-7B-Instruct-AWQ를 서빙하고, ngrok으로 로컬 PC에서 접근할 수 있도록 합니다.

**사전 준비**:
1. Runtime → Change runtime type → **T4 GPU** 선택
2. [ngrok.com](https://ngrok.com)에서 무료 계정 생성 후 Auth Token 복사

## 셀 1 — vLLM 설치

In [ ]:
!pip install vllm

## 셀 2 — vLLM 서버 시작 (백그라운드)

In [ ]:
import subprocess
import time
import requests

proc = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "--quantization", "awq",
    "--port", "8000",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--enforce-eager",
])
print(f"vLLM 서버 PID: {proc.pid}")

# 서버 준비될 때까지 대기 (최대 5분)
print("vLLM 모델 로딩 대기 중...")
for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        if r.status_code == 200:
            print(f"✅ vLLM 준비 완료 (약 {i*5}초 소요)")
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print("⚠️ 5분 경과. 서버가 아직 준비되지 않았을 수 있습니다.")

## 셀 3 — ngrok 터널 생성

아래 `YOUR_NGROK_TOKEN`을 실제 토큰으로 교체하세요.

In [ ]:
!pip install pyngrok

from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_TOKEN")
tunnel = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  로컬 PC에서 아래 환경변수를 설정하세요:")
print(f"  set VLLM_BASE_URL={tunnel.public_url}/v1")
print(f"{'='*60}")

## 셀 4 — 헬스체크

모델 로딩에 약 30초 소요됩니다. 정상적으로 모델 목록이 출력되면 준비 완료.

In [ ]:
import requests
import time

time.sleep(30)

r = requests.get(f"{tunnel.public_url}/v1/models")
print(r.json())